# Deploy Grants

Grant the DICOMweb app's service principal access to:
- Lakebase tables (`dicom_frames`, `instance_paths`, `endpoint_metrics`)
- Unity Catalog (catalog, schema, table, volume)

In [0]:
import os
_nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_repo_root = os.path.normpath(os.path.join("/Workspace" + os.path.dirname(_nb_ctx.notebookPath().get()), ".."))

%pip install --quiet -r {_repo_root}/requirements.txt
dbutils.library.restartPython()

In [0]:
%run ./config/proxy_prep

In [0]:
sql_warehouse_id, table, volume = init_widgets(show_volume=True)
init_env()

In [ ]:
app_gateway_name = "pixels-dicomweb-gateway"
app_viewer_name = "pixels-dicomweb"
lakebase_instance_name = dbutils.widgets.get("lakebase_instance_name")

# Get Service Principal and Reconnect to Lakebase

In [0]:
from dbx.pixels.lakebase import LakebaseUtils, parse_uc_table_name

# Reconnect to existing Lakebase instance (do NOT create — already done by deploy-lakebase)
lb_utils = LakebaseUtils(instance_name=lakebase_instance_name, uc_table_name=table, create_instance=False)
_lb_schema = lb_utils.schema

_uc_catalog, _uc_schema, _uc_table = parse_uc_table_name(table)

# Grant BOTH the gateway and viewer service principals. Both apps install the
# pixels wheel from the UC volume on first deploy and need USE_CATALOG +
# volume read; granting only the gateway SP breaks viewer deploys.
sp_ids = {
    app_gateway_name: w.apps.get(app_gateway_name).service_principal_client_id,
    app_viewer_name: w.apps.get(app_viewer_name).service_principal_client_id,
}

for name, sp in sp_ids.items():
    lb_utils.get_or_create_sp_role(sp)
    print(f"{name} SP: {sp}")

# Lakebase Grants

In [0]:
# Lakebase grants — apply to both gateway and viewer SPs
for sp in sp_ids.values():
    lb_utils.execute_query(
        f'GRANT SELECT, INSERT, UPDATE, DELETE ON {_lb_schema}.dicom_frames TO "{sp}"'
    )

    # instance_paths is created by the synced table pipeline — grant only if sync has completed
    try:
        lb_utils.execute_query(
            f'GRANT SELECT, INSERT, UPDATE, DELETE ON {_lb_schema}.instance_paths TO "{sp}"'
        )
    except Exception as e:
        if "does not exist" in str(e):
            print(
                f"⚠ Skipping instance_paths grant for {sp} — table not yet synced (pipeline may still be running)"
            )
        else:
            raise

    lb_utils.execute_query(
        f'GRANT SELECT, INSERT, UPDATE, DELETE ON {_lb_schema}.endpoint_metrics TO "{sp}"'
    )
    lb_utils.execute_query(f'GRANT USAGE ON SCHEMA {_lb_schema} TO "{sp}"')
    lb_utils.execute_query(
        f'ALTER DEFAULT PRIVILEGES IN SCHEMA {_lb_schema} GRANT SELECT, INSERT, UPDATE, DELETE ON TABLES TO "{sp}"'
    )

# Index for fast Study Instance UID lookups on the synced table (SP-independent)
try:
    lb_utils.execute_query(
        f"CREATE INDEX IF NOT EXISTS idx_instance_paths_study ON {_lb_schema}.instance_paths (study_instance_uid)"
    )
except Exception as e:
    if "does not exist" in str(e):
        print(
            "⚠ Skipping instance_paths index — table not yet synced (pipeline may still be running)"
        )
    else:
        raise

print("✓ Lakebase grants applied")

# Unity Catalog Grants

In [0]:
from databricks.sdk.service import catalog

for sp in sp_ids.values():
    w.grants.update(
        full_name=_uc_catalog,
        securable_type="catalog",
        changes=[catalog.PermissionsChange(add=[catalog.Privilege.USE_CATALOG], principal=sp)],
    )
    w.grants.update(
        full_name=f"{_uc_catalog}.{_uc_schema}",
        securable_type="schema",
        changes=[catalog.PermissionsChange(add=[catalog.Privilege.USE_SCHEMA], principal=sp)],
    )
    w.grants.update(
        full_name=table,
        securable_type="table",
        changes=[catalog.PermissionsChange(add=[catalog.Privilege.ALL_PRIVILEGES], principal=sp)],
    )
    w.grants.update(
        full_name=volume,
        securable_type="volume",
        changes=[catalog.PermissionsChange(add=[catalog.Privilege.ALL_PRIVILEGES], principal=sp)],
    )

print("✅ All grants applied")